# Avance 6 - Visualizacion cualitativa del mejor modelo

**Mejor modelo: MonoIIT** (AbsRel 0.0554, el mas bajo de los 6 y consistente con la referencia del Dr. Espinosa).

Para frames representativos del *split* oficial se muestran, lado a lado:
- La imagen **original** y su mapa de profundidad
- Cada **enhancement** (retinex, endolmspec, iat) y su mapa de profundidad

Asi se ve cualitativamente el efecto de cada correccion sobre la estimacion de profundidad.

In [ ]:
from pathlib import Path
import sys, subprocess

IN_COLAB = "google.colab" in sys.modules
if not IN_COLAB:
    try:
        import google.colab; IN_COLAB = True
    except ImportError:
        IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    BASE         = Path("/content/drive/MyDrive/proyecto_integrador")
    SCARED_ROOT  = BASE / "scared_raw"
    EDAM_PATH    = BASE / "Endo-Depth-and-Motion"
    LMSPEC_PATH  = BASE / "EndoLMSPEC"
    IAT_PATH     = BASE / "EndoViT"
    MONOVIT_PATH = BASE / "MonoViT"
    STTN_PATH    = BASE / "Endo-STTN"
    W            = BASE / "scared weights"
    REPO_ROOT    = Path("/content/repo_52")
    if not REPO_ROOT.exists():
        subprocess.check_call(["git","clone","--depth=1",
            "https://github.com/jmtoral/proyecto_integrador_52.git", str(REPO_ROOT)])
    else:
        subprocess.check_call(["git","-C",str(REPO_ROOT),"fetch","origin"])
        subprocess.check_call(["git","-C",str(REPO_ROOT),"reset","--hard","origin/main"])
    SPLIT_FILE = REPO_ROOT / "data" / "splits" / "endovis" / "test_files.txt"
else:
    BASE         = Path("E:/scared_wights_complete/scared weights")
    SCARED_ROOT  = Path("D:/Proyecto_Integrador/Corrreccion_Luz/data/scared_raw")
    EDAM_PATH    = Path("E:/Endo-Depth-and-Motion")
    LMSPEC_PATH  = Path("E:/EndoLMSPEC")
    IAT_PATH     = Path("E:/EndoVit")
    MONOVIT_PATH = Path("E:/MonoViT")
    STTN_PATH    = Path("E:/Endo-STTN")
    W            = BASE
    REPO_ROOT    = Path(r"d:\Proyecto_Integrador\Corrreccion_Luz")
    SPLIT_FILE   = REPO_ROOT / "data" / "splits" / "endovis" / "test_files.txt"

W_MONOIIT      = W / "monoIIT_weights" / "trained-winner-weights"
LMSPEC_WEIGHTS = LMSPEC_PATH / "checkpoint" / "main_net" / "model_256_combined_SSIM5_1.pth"
IAT_WEIGHTS    = IAT_PATH / "Endo4IE" / "best_Epoch50_laplacian_histogan_loss.pth"
STTN_WEIGHTS   = STTN_PATH / "release_model" / "pretrained_model" / "gen_00009.pth"
NPZ_CACHE      = BASE / "split_frames.npz"

def load_split(sf):
    items=[]
    with open(sf) as f:
        for line in f:
            line=line.strip()
            if not line: continue
            folder, fid, _ = line.split()
            ds, kf = folder.split("/")
            items.append(("dataset_"+ds.replace("dataset",""), "keyframe_"+kf.replace("keyframe",""), int(fid)))
    return items
SPLIT_ITEMS = load_split(SPLIT_FILE)
from collections import defaultdict
SPLIT_BY_KF = defaultdict(list)
for ds,kf,fid in SPLIT_ITEMS: SPLIT_BY_KF[(ds,kf)].append(fid)
print(f"{'Colab' if IN_COLAB else 'Local'} | split {len(SPLIT_ITEMS)} frames")

In [ ]:
import numpy as np
# Cargar imagenes + GT del npz cacheado
def _key(ds,kf,fid): return f"{ds}|{kf}|{fid}"
SPLIT_DATA = {}
_npz = np.load(NPZ_CACHE, allow_pickle=True)
for ds,kf,fid in SPLIT_ITEMS:
    k=_key(ds,kf,fid); ik,gk="img_"+k,"gt_"+k
    if ik in _npz.files:
        gt=_npz[gk] if gk in _npz.files else None
        if gt is not None and gt.size==1 and np.isnan(gt).all(): gt=None
        SPLIT_DATA[k]=(_npz[ik], gt)
def load_split_frame(ds,kf,fid):
    return SPLIT_DATA.get(_key(ds,kf,fid),(None,None))
print(f"Frames en memoria: {len(SPLIT_DATA)}")

In [ ]:
import torch, importlib.util as _ilu, types
import torch.nn as nn
from collections import OrderedDict
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

_NETDIR = MONOVIT_PATH / "networks"
for _k in list(sys.modules):
    if _k=="networks" or _k.startswith("networks."): del sys.modules[_k]
_pkg=types.ModuleType("networks"); _pkg.__path__=[str(_NETDIR)]; sys.modules["networks"]=_pkg
def _ls(name,fn):
    sp=_ilu.spec_from_file_location(f"networks.{name}",str(_NETDIR/fn))
    m=_ilu.module_from_spec(sp); sys.modules[f"networks.{name}"]=m; sp.loader.exec_module(m); setattr(_pkg,name,m); return m
_ls("hr_layers","hr_layers.py")
mpvit_small=_ls("mpvit","mpvit.py").mpvit_small

def _up(x): return nn.functional.interpolate(x,scale_factor=2,mode="nearest")
class _C3(nn.Module):
    def __init__(s,i,o): super().__init__(); s.pad=nn.ReflectionPad2d(1); s.conv=nn.Conv2d(int(i),int(o),3)
    def forward(s,x): return s.conv(s.pad(x))
class _CB(nn.Module):
    def __init__(s,i,o): super().__init__(); s.conv=_C3(i,o); s.nl=nn.ELU(inplace=False)
    def forward(s,x): return s.nl(s.conv(x))
class DecSimple(nn.Module):
    def __init__(s,nce,ncd,scales=range(4),skips=True):
        super().__init__(); s.scales=list(scales); s.skips=skips
        s.nce=np.array(nce); s.ncd=np.array(ncd); s.convs=OrderedDict()
        for i in range(4,-1,-1):
            ci=s.nce[-1] if i==4 else s.ncd[i+1]
            s.convs[("upconv",i,0)]=_CB(ci,s.ncd[i]); ci=s.ncd[i]
            if s.skips and i>0: ci+=s.nce[i-1]
            s.convs[("upconv",i,1)]=_CB(ci,s.ncd[i])
        for sc in s.scales: s.convs[("dispconv",sc)]=_C3(s.ncd[sc],1)
        s.decoder=nn.ModuleList(list(s.convs.values())); s.sig=nn.Sigmoid()
    def forward(s,feats):
        out={}; x=feats[-1]
        for i in range(4,-1,-1):
            x=s.convs[("upconv",i,0)](x); x=[_up(x)]
            if s.skips and i>0: x+=[feats[i-1]]
            x=torch.cat(x,1); x=s.convs[("upconv",i,1)](x)
            if i in s.scales: out[("disp",i)]=s.sig(s.convs[("dispconv",i)](x))
        return out

# Cargar MonoIIT (encoder mpvit + decoder simple, num_ch_dec inferido del checkpoint)
enc = mpvit_small(); enc.num_ch_enc=[64,128,216,288,288]
_ed = torch.load(W_MONOIIT/"encoder.pth", map_location=DEVICE)
MH, MW = _ed.get("height",192), _ed.get("width",640)
enc.load_state_dict({k:v for k,v in _ed.items() if k in enc.state_dict()}); enc.to(DEVICE).eval()
_sd = torch.load(W_MONOIIT/"depth.pth", map_location=DEVICE)
_dims={}
for k,v in _sd.items():
    if k.endswith("conv.conv.weight") and ".decoder." in k:
        _dims[int(k.split(".decoder.")[1].split(".")[0])]=v.shape[0]
_ncd=[_dims[8],_dims[6],_dims[4],_dims[2],_dims[0]]
dec = DecSimple([64,128,216,288,288], _ncd)
_r = dec.load_state_dict(_sd, strict=False); dec.to(DEVICE).eval()
print(f"MonoIIT cargado {MH}x{MW} | num_ch_dec={_ncd} | missing={len(_r.missing_keys)} unexpected={len(_r.unexpected_keys)}")

import cv2, PIL.Image as pil
from torchvision import transforms
def predict_depth(img, max_depth=150.0, min_depth=0.1):
    H,W_=img.shape[:2]
    t=transforms.ToTensor()(pil.fromarray(img).resize((MW,MH),pil.LANCZOS)).unsqueeze(0).to(DEVICE)
    with torch.no_grad(): out=dec(enc(t))
    disp=out[("disp",0)].squeeze().detach().cpu().numpy()
    sd=(1.0/max_depth)+((1.0/min_depth)-(1.0/max_depth))*disp
    sd=cv2.resize(sd,(W_,H)); return 1.0/sd

In [ ]:
# Cargar los enhancements (mismo codigo que el notebook principal)
import torchvision.transforms as T
subprocess.check_call([sys.executable,"-m","pip","install","-q","IQA_pytorch","path"])

def _load_endolmspec(p, device):
    _orig=sys.path.copy()
    clean=[str(p)]+[x for x in sys.path if "EndoSLAM" not in x and "endosfm" not in x.lower()
                    and "HADepth" not in x and "EndoViT" not in x and "EndoVit" not in x]
    for k in list(sys.modules):
        if k in ("utils","generator","unet") or k.startswith("utils."): del sys.modules[k]
    try:
        sys.path=clean
        sp=_ilu.spec_from_file_location("generator", p/"generator.py")
        mod=_ilu.module_from_spec(sp); sp.loader.exec_module(mod); G=mod.Generator
    finally: sys.path=_orig
    return G(n_channels=3, device=device, bilinear=False)
lmspec_net=_load_endolmspec(LMSPEC_PATH, DEVICE)
lmspec_net.load_state_dict(torch.load(LMSPEC_WEIGHTS, map_location=DEVICE)); lmspec_net.to(DEVICE).eval()

for k in list(sys.modules):
    if k=="utils" or k.startswith("utils."): del sys.modules[k]
sys.modules["imp"]=types.ModuleType("imp")
_sp=_ilu.spec_from_file_location("IAT_main_a5", IAT_PATH/"experiments"/"model"/"IAT_main.py")
_im=_ilu.module_from_spec(_sp)
if str(IAT_PATH/"experiments") not in sys.path: sys.path.insert(0,str(IAT_PATH/"experiments"))
_sp.loader.exec_module(_im)
iat_net=_im.IAT(in_dim=3, with_global=True, type="exp")
iat_net.load_state_dict(torch.load(IAT_WEIGHTS, map_location=DEVICE)); iat_net.to(DEVICE).eval()

def correct_none(img): return img
def correct_retinex(img, sigma=30):
    f=img.astype(np.float32)+1.0; r=np.zeros_like(f)
    for c in range(3):
        b=cv2.GaussianBlur(f[:,:,c],(0,0),sigma); r[:,:,c]=np.log(f[:,:,c])-np.log(b+1.0)
    r-=r.min(); return (r/(r.max()+1e-8)*255).astype(np.uint8)
def correct_endolmspec(img):
    t=T.ToTensor()(img).to(DEVICE)
    with torch.no_grad(): _,o=lmspec_net(t)
    return (o["subnet_16"][0].cpu().clamp(0,1).permute(1,2,0).numpy()*255).astype(np.uint8)
def correct_iat(img):
    t=torch.from_numpy(img.astype(np.float32)/255).permute(2,0,1).unsqueeze(0).to(DEVICE)
    with torch.no_grad(): _,_,e=iat_net(t)
    return (e[0].cpu().clamp(0,1).permute(1,2,0).numpy()*255).astype(np.uint8)

CORRECTIONS={"none":correct_none,"retinex":correct_retinex,"endolmspec":correct_endolmspec,"iat":correct_iat}
print("Enhancements:", list(CORRECTIONS.keys()))
print("(endosttn se omite en esta visualizacion por ser temporal/pesado; se puede anadir si se desea)")

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

# Elegir frames representativos de keyframes distintos
_kfs = list(SPLIT_BY_KF.keys())
VIZ = [(_kfs[0][0], _kfs[0][1], SPLIT_BY_KF[_kfs[0]][len(SPLIT_BY_KF[_kfs[0]])//2]),
       (_kfs[len(_kfs)//2][0], _kfs[len(_kfs)//2][1], SPLIT_BY_KF[_kfs[len(_kfs)//2]][0])]

methods = list(CORRECTIONS.keys())
for (ds,kf,fid) in VIZ:
    img,_ = load_split_frame(ds,kf,fid)
    if img is None: continue
    fig, axes = plt.subplots(2, len(methods), figsize=(4*len(methods), 7))
    fig.suptitle(f"MonoIIT - {ds}/{kf} frame {fid}", fontsize=13, fontweight="bold")
    for j, m in enumerate(methods):
        corr = CORRECTIONS[m](img)
        depth = predict_depth(corr)
        axes[0,j].imshow(corr); axes[0,j].set_title(m, fontsize=11); axes[0,j].axis("off")
        axes[1,j].imshow(depth, cmap="magma"); axes[1,j].axis("off")
        if j==0:
            axes[0,j].set_ylabel("imagen", fontsize=10)
            axes[1,j].set_ylabel("profundidad", fontsize=10)
    plt.tight_layout(); plt.show()